In [1]:
# import os
# os.environ['JAX_ENABLE_X64'] = 'True'
# os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'true'
# os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.3'
from __future__ import annotations

import os
import ctypes

# 1.
cuda_lib = "/apps/cuda/cuda-13.0/lib64/libcudart.so.13"
cupti_lib = "/apps/cuda/cuda-13.0/extras/CUPTI/lib64/libcupti.so.13"
cudnn_lib = "/apps/cuda/cudnn-linux-x86_64-9.14.0.64_cuda13/lib/libcudnn.so.9"

# 2.
try:
    ctypes.CDLL(cuda_lib)
    ctypes.CDLL(cupti_lib)
    ctypes.CDLL(cudnn_lib)
except Exception as e:
    print(f"wrong check: {e}")

os.environ["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir=/apps/cuda/cuda-13.0"

import time
from dataclasses import field
from functools import partial

import pathlib
import sys
# script_dir = pathlib.Path('.').parent.absolute()
# sys.path.insert(0, str(script_dir / 'ong-python-toolbox' / 'src'))
#
# from ong.utils import gpu_init
#
# import jax
# import numpy as np
#



from jax import numpy as jnp
from matplotlib import pyplot as plt
from scipy import interpolate
from scipy.optimize import minimize, Bounds

import pathlib
_NOTEBOOK_DIR = pathlib.Path.cwd().resolve()
_REPO_ROOT = next((p for p in (_NOTEBOOK_DIR, *_NOTEBOOK_DIR.parents) if (p / 'external').exists()), _NOTEBOOK_DIR)
_EXTERNAL_DIR = _REPO_ROOT / 'external'
_ONG_SRC = _EXTERNAL_DIR / 'ong-python-toolbox' / 'src'
for p in (str(_EXTERNAL_DIR), str(_ONG_SRC)):
    if p not in sys.path:
        sys.path.insert(0, p)

from ong import models
from ong.measurements import load_g652d_raman, load_corning_smf_28_raman
from ong.models import raman_solver, FibreSpanSetupAdvanced, isrs_gn_model_integral, egn
from ong.utils import struct, console
from sibase import Value
from ong.utils.misc import timer
import data as fibre_data
from scipy.constants import c, pi
from rich import progress

dBm = lambda x: 10 * np.log10(x/1e-3)
idBm = lambda x: 10 ** (x / 10)*1e-3 
dB = lambda x: 10 * np.log10(x)
idB = lambda x: 10 ** (x / 10)


@struct.setupclass
class SystemRequiredSetup:
    name: str
    symbol_rate: Value
    seq_length: int
    delta_g: float


@struct.setupclass
class SystemSetup(
    models.GNSetup,
    SystemRequiredSetup  # this has to be last
):
    description: str = ''
    rng_seed: int = field(default_factory=lambda: time.time_ns())
    roll_off: float = 0.01
    samples_per_symbol = 2
    modulation_name: str = 'gaussian'
    num_polarisation = 2
    PMD_parameter = 0
    DGD_parameter = 0
    polarisation_rotation = False

    @property
    def prng_key(self):
        return jax.random.PRNGKey(self.rng_seed)

    @property
    def sample_rate(self):
        return self.samples_per_symbol * self.symbol_rate


@struct.setupclass
class DynPowerSystemSetup(SystemSetup):
    ch_power_dBm: struct.ArrayLike
    noise_figure: struct.ArrayLike
    snr_trx: struct.ArrayLike
    channel_idx: struct.ArrayLike

    @property
    def beta4_j(self) -> jnp.ndarray:
        return jnp.array([span.beta4 for span in self.spans])

In [2]:
from ong.models.raman_fitting import get_power_profile_fit

import jax
import numpy as np
from jax import numpy as jnp
from jax.numpy import exp, sum, sqrt, log, zeros, abs, sign, arcsinh, arctan, nansum, mean, nan_to_num
from scipy.constants import pi, c

_INDICES = (jnp.arange(4)[:, None] >> jnp.arange(2)) & 1
_INDICES_COI = (jnp.arange(16)[:, None] >> jnp.arange(4)) & 1
_INDICES_FWM = (jnp.arange(64)[:, None] >> jnp.arange(6)) & 1

In [3]:
BANDS = {
    'O': {
        'TransceiverSNR': 20,
        'noise_figure': 5,
        'p_launch': 23
    },
    'E': {
        'TransceiverSNR': 18.5,
        'noise_figure': 6,
        'p_launch': 15.57
    },
    'S': {
        'TransceiverSNR': 20,
        'noise_figure': 7,
        'p_launch': 14.03
    },
    'C': {
        'TransceiverSNR': 23.5,
        'noise_figure': 5,
        'p_launch': 20.69
    },
    'L': {
        'TransceiverSNR': 22,
        'noise_figure': 5.5,
        'p_launch': 21.9
    },
}

In [4]:
ch_spacing = 50e9  # [Hz]
ch_bw = 48e9  # [Hz]
num_channels = 1111
reflambda = 1419.4e-9  # [m]
length = 80  # [km]
Nspans = 1
detail = False
fibre = 'min'
gamma = 2  # [1/W/km]

lop = -2  # [dBm]

In [5]:
chs = jnp.arange(num_channels) - (num_channels - 1) / 2
ch_lambda = c / (chs * ch_spacing + c / reflambda)

# longest L
ll = 1625e-9
idx = ch_lambda >= ll
L_band_bound = jnp.where(idx)[0]

# longest C and shortest L
cl = 1566e-9
ls = 1573e-9
idx = (ch_lambda >= cl) & (ch_lambda <= ls)
CL_band_gap = jnp.where(idx)[0]

# longest S and shortest C
sl = 1526e-9
cs = 1530e-9
idx = (ch_lambda >= sl) & (ch_lambda <= cs)
SC_band_gap = jnp.where(idx)[0]

# longest E and shortest S
el = 1464e-9
ss = 1470e-9
idx = (ch_lambda >= el) & (ch_lambda <= ss)
ES_band_gap = jnp.where(idx)[0]

# longest O and shortest E
ol = 1358e-9
es = 1405e-9
idx = (ch_lambda >= ol) & (ch_lambda <= es)
OE_band_gap = jnp.where(idx)[0]

# shortest O 
os = 1260e-9
idx = ch_lambda <= os
O_band_bound = jnp.where(idx)[0]

ch_gaps = jnp.concatenate([L_band_bound, CL_band_gap, SC_band_gap, ES_band_gap, OE_band_gap, O_band_bound], axis=0)

ch_idx_oband = (ch_lambda <= ol) & (ch_lambda >= os)
ch_idx_eband = (ch_lambda <= el) & (ch_lambda >= es)
ch_idx_sband = (ch_lambda <= sl) & (ch_lambda >= ss)
ch_idx_cband = (ch_lambda <= cl) & (ch_lambda >= cs)
ch_idx_lband = (ch_lambda <= ll) & (ch_lambda >= ls)

channel_idx = ch_idx_oband | ch_idx_eband | ch_idx_sband | ch_idx_cband | ch_idx_lband
lambda_upper = max(ch_lambda[channel_idx])
lambda_lower = min(ch_lambda[channel_idx])

In [6]:
print(f'Simulated bandwidth from {lambda_lower * 1e9:.1f} nm to {lambda_upper * 1e9:.1f} nm with {sum(channel_idx)} channels') 

Simulated bandwidth from 1260.1 nm to 1624.8 nm with 878 channels


In [7]:
# Launch power
P_channel = -jnp.inf * jnp.ones((num_channels))  # [dBm]
P_channel = P_channel.at[ch_idx_oband].set(lop)
P_channel = P_channel.at[ch_idx_eband].set(lop)
P_channel = P_channel.at[ch_idx_sband].set(lop)
P_channel = P_channel.at[ch_idx_cband].set(lop)
P_channel = P_channel.at[ch_idx_lband].set(lop)

# Noise figure
nf = jnp.zeros(num_channels)  # [dB]

nf = nf.at[ch_idx_oband].set(BANDS['O']['noise_figure'])
nf = nf.at[ch_idx_eband].set(BANDS['E']['noise_figure'])
nf = nf.at[ch_idx_sband].set(BANDS['S']['noise_figure'])
nf = nf.at[ch_idx_cband].set(BANDS['C']['noise_figure'])
nf = nf.at[ch_idx_lband].set(BANDS['L']['noise_figure'])

# Transceiver SNR
snr_trx = jnp.zeros(num_channels)  # [dB]

snr_trx = snr_trx.at[ch_idx_oband].set(BANDS['O']['TransceiverSNR'])
snr_trx = snr_trx.at[ch_idx_eband].set(BANDS['E']['TransceiverSNR'])
snr_trx = snr_trx.at[ch_idx_sband].set(BANDS['S']['TransceiverSNR'])
snr_trx = snr_trx.at[ch_idx_cband].set(BANDS['C']['TransceiverSNR'])
snr_trx = snr_trx.at[ch_idx_lband].set(BANDS['L']['TransceiverSNR'])

In [8]:
dispersion_fit_wavelength = jnp.array([lambda_lower, reflambda, lambda_upper])
fibre_Furukawa_Allwave_ULL = FibreSpanSetupAdvanced(
    length=f'{length}km', dispersion_order=2, dispersion_ref_lambda=reflambda,  # can I just change the dispersion order here?
    dispersion_fit_points=dispersion_fit_wavelength,  # more than 3 points to enable using all wavelength fitting
    # attenuation_profile=data.fit_attenuation(fibre),
    # raman_profile=load_g652d_raman(),

    
    attenuation_profile=fibre_data.fit_attenuation(fibre),
    dispersion_profile=fibre_data.get_dispersion(fibre),

    raman_profile=load_corning_smf_28_raman(),
    # dispersion_profile=data.get_dispersion(fibre),
    effective_area_profile=(jnp.array([1310e-9, 1550e-9]), jnp.array([66.476e-12, 86.59e-12])),
    nonlinear_coeff_profile=(jnp.array([1310e-9]), jnp.array([gamma * 1e-3]))
)

setup = DynPowerSystemSetup(
    spans=[fibre_Furukawa_Allwave_ULL],
    delta_g=1e-7,
    seq_length=2 ** 16,
    symbol_rate=f'{ch_bw * 1e-9}GBaud',
    num_channels=num_channels,
    ch_bandwidth=f'{ch_bw * 1e-9}GHz',
    ch_spacing=f'{ch_spacing * 1e-9}GHz',
    ref_lambda=reflambda,
    ch_power_dBm=P_channel,
    name=f"",
    description="Simulating O-band",
    roll_off=0.0001,
    noise_figure=nf,
    modulation_name='16QAM',
    snr_trx=snr_trx,
    channel_idx=channel_idx,
)

In [ ]:
def calc_NSR_link(setup, Nspans, mask, ch_idx_oband=None):
    j = 0
    # chs = jnp.arange(setup.num_channels)
    chs = setup.channel_idx
    gamma_i = jnp.array(setup.spans[j].nonlinear_coeff_at(setup.ch_lambda_ij[chs, j]))
    beta2_j = jnp.array([setup.spans[j].beta2])
    beta3_j = jnp.array([setup.spans[j].beta3])
    beta4_j = jnp.array([setup.spans[j].beta4])
    Aeff_i = jnp.array(setup.spans[j].A_eff_at(setup.ch_lambda_ij[chs, j]))

    att = setup.attenuation_ij[chs, j]

    l = setup.length_j[j]
    z = jnp.arange(0, l, 1000)

    ch_power_W_i = setup.ch_power_W_ij[chs, :] * mask

    _, power_evo = raman_solver.solve_isrs_evolution(
        ch_centre_i=setup.ch_centre_ij[chs, j],
        A_eff=Aeff_i,
        raman_profile=setup.raman_profile_j[j],
        length=setup.length_j[j],
        attenuation_i=setup.attenuation_ij[chs, j],
        ch_power_W_i=ch_power_W_i[:, j],
        ref_lambda=setup.ref_lambda,
        zspan=z,
    )

    fit_params = get_power_profile_fit(length_j=setup.length_j,
                                power_evo_j=power_evo[None, :, :],
                                ch_centre_ij=setup.ch_centre_ij,
                                ch_power_W_ij=ch_power_W_i,
                                attenuation_ij=setup.attenuation_ij,
                                raman_gain_slope_j=setup.raman_gain_slope_j,
                                zspan=z)[:, :, None]

    a = fit_params[:, 0]
    a_bar = fit_params[:, 1]
    Cr = fit_params[:, 2]

    L = setup.length_j
    P_ij = ch_power_W_i
    Ptot = jnp.sum(P_ij, axis=0)
    gamma_ij = gamma_i[:, None]
    if gamma_ij.ndim == 1:
        gamma_ij = jnp.repeat(gamma_ij[None], num_channels, axis=1)
    fi = setup.ch_centre_ij
    Bch = setup.ch_bandwidth_ij
    mean_att_i = jnp.mean(a, axis=1)  # average attenuation coefficent for channel i
    mean_L = jnp.mean(L)  # average fiber length

    a_i = a[:, j:j+1]  # \alpha of COI in fiber span j
    a_k = jnp.transpose(a[:, j:j+1])  # \alpha of INT in fiber span j
    a_bar_i = a_bar[:, j:j+1]  # \bar{\alpha} of COI in fiber span j
    a_bar_k = jnp.transpose(a_bar[:, j:j+1])  # \bar{\alpha} of INT in fiber span j
    f_i = fi[chs, j:j+1]  # f_i of COI in fiber span j
    f_k = jnp.transpose(fi[chs, j:j+1])  # f_k of INT in fiber span j
    B_i = Bch[chs, j:j+1]  # B_i of COI in fiber span j
    B_k = jnp.transpose(Bch[chs, j:j+1])  # B_k of INT in fiber span j
    Cr_i = Cr[:, j:j+1]  # Cr of COI in fiber span j
    Cr_k = jnp.transpose(Cr[:, j:j+1])  # Cr of INT in fiber span j
    P_i = P_ij[:, j:j+1]  # P_i of COI in fiber span j
    P_k = jnp.transpose(P_ij[:, j:j+1])  # P_k of INT in fiber span j

    # \phi_i  of COI in fiber span j
    phi_i = -4 * pi ** 2 * (beta2_j[j] + pi * beta3_j[j] * (f_i + f_i) + 2 * pi ** 2 * beta4_j[j] * f_i ** 2)

    # \phi_ik of COI-INT pair in fiber span j
    phi_ik = -4 * pi ** 2 * (f_k - f_i) * (beta2_j[j] + pi * beta3_j[j] * (f_i + f_k) + 2 / 3 * pi ** 2 * beta4_j[j] * (f_i ** 2 + f_i * f_k + f_k ** 2))

    Tf_i = (a_i + a_bar_i - f_i * Ptot[j] * Cr_i) ** 2  # T_i of COI in fiber span j
    Tf_k = (a_k + a_bar_k - f_k * Ptot[j] * Cr_k) ** 2  # T_k of INT in fiber span j

    Tf_i = -((Ptot[j] * Cr_i) / a_bar_i) * f_i
    Tf_k = -((Ptot[j] * Cr_k) / a_bar_k) * f_k

    T_i = 1 + Tf_i
    T_k = 1 + Tf_k
    @jax.jit
    def _eta_GN_SPM(phi_i, B_i, a, a_bar, gamma, Tf, T, L):
        def _fun(x):
            l, l_line = x
            alpha = a + a_bar * l
            alpha_tilde = (alpha * (1 - exp(-alpha * L))) / (1 - exp(-alpha * L) - alpha * L * exp(-alpha * L))
            alpha_line = a + a_bar * l_line
            alpha_tilde_line = (alpha_line * (1 - exp(-alpha_line * L))) / (1 - exp(-alpha_line * L) - alpha_line * L * exp(-alpha_line * L))
            T_tilde = T * ((-Tf / T) ** l)
            T_tilde_line = T * ((-Tf / T) ** l_line)
            kappa = ((1 - exp(-alpha * L)) ** 2) / (1 - exp(-alpha * L) - alpha * L * exp(-alpha * L))
            kappa_line = ((1 - exp(-alpha_line * L)) ** 2) / (1 - exp(-alpha_line * L) - alpha_line * L * exp(-alpha_line * L))
            idx = (phi_i == 0)
            return (nan_to_num(
                (16 / 27) * gamma ** 2 / B_i ** 2
                * ((2 * kappa * kappa_line * pi * T_tilde * T_tilde_line) / (phi_i * (alpha_tilde + alpha_tilde_line)))
                * (arcsinh(3 * phi_i * B_i ** 2 / (8 * pi * alpha_tilde)) + arcsinh(3 * phi_i * B_i ** 2 / (8 * pi * alpha_tilde_line))),
                nan=0,
                posinf=0,
                neginf=0,
            ) + idx * ((16 / 27) * gamma ** 2 * (kappa * kappa_line * T_tilde * T_tilde_line * (3 / (4 * alpha_tilde * alpha_tilde_line))))).squeeze()

        return jax.vmap(_fun)(_INDICES).sum(axis=0)


    @jax.jit
    def _eta_GN_XPM(Pi, Pk, phi_ik, B_i, B_k, a, a_bar, gamma, Tf, T, L):
        def _fun(x):
            l, l_line = x
            alpha = a + a_bar * l
            alpha_tilde = alpha * (1 - exp(-alpha * L)) / (1 - exp(-alpha * L) - alpha * L * exp(-alpha * L))
            alpha_line = a + a_bar * l_line
            alpha_tilde_line = alpha_line * (1 - exp(-alpha_line * L)) / (1 - exp(-alpha_line * L) - alpha_line * L * exp(-alpha_line * L))
            kappa = (1 - exp(-alpha * L)) ** 2 / (1 - exp(-alpha * L) - alpha * L * exp(-alpha * L))
            kappa_line = (1 - exp(-alpha_line * L)) ** 2 / (1 - exp(-alpha_line * L) - alpha_line * L * exp(-alpha_line * L))
            T_tilde = T * ((-Tf / T) ** l)
            T_tilde_line = T * ((-Tf / T) ** l_line)
            return 32 / 27 * jnp.sum(
                nan_to_num(
                    (Pk / Pi) ** 2 * gamma ** 2 / B_k
                    * (kappa * kappa_line * 2 * T_tilde * T_tilde_line / (phi_ik * (alpha_tilde + alpha_tilde_line)))
                    * (arctan(phi_ik * B_i / (2 * alpha_tilde)) + arctan(phi_ik * B_i / (2 * alpha_tilde_line))),
                    nan=0,
                    posinf=0,
                    neginf=0,
                ),
                axis=1,
            ).squeeze()

        return jax.vmap(_fun)(_INDICES).sum(axis=0)


    # def _FWM_idx(f):
    #     freqs = f.squeeze()

    #     f_i = freqs[:, None, None, None]
    #     f_j = freqs[None, :, None, None]
    #     f_k = freqs[None, None, :, None]
    #     f_m = freqs[None, None, None, :]
    #     cond = ((f_j + f_k - f_m) == f_i) & (f_j != f_i) & (f_k != f_m) & (f_k != f_i) & (f_j != f_m) & (f_j <= f_k)
    #     i_idx, j_idx, k_idx, m_idx = jnp.nonzero(cond)
    #     idx_flat = jnp.stack([i_idx, j_idx, k_idx, m_idx], axis=-1)
    #     counts = jnp.bincount(i_idx, length=freqs.size)

    #     n, K, M = counts.size, idx_flat.shape[0], int(counts.max())
    #     s = jnp.concatenate([jnp.array([0]), jnp.cumsum(counts)[:-1]])
    #     ch = jnp.repeat(jnp.arange(n), counts)
    #     r = jnp.arange(K) - s[ch]
    #     idx_pad = jnp.zeros((n, M, idx_flat.shape[1]), idx_flat.dtype).at[ch, r].set(idx_flat)
    #     valid = jnp.zeros((n, M)).at[ch, r].set(True)

    #     return idx_pad, valid


    def _FWM_idx(f):
        freqs = f.squeeze()
        n = freqs.size

        all_idx = []
        counts = []

        for i in range(n):
            fi = freqs[i]

            # Build only (j,k) grid for this i
            j_idx, k_idx = jnp.meshgrid(jnp.arange(n), jnp.arange(n), indexing="ij")
            j_idx = j_idx.ravel()
            k_idx = k_idx.ravel()

            # Solve for m from frequency relation: f_m = f_j + f_k - f_i
            target_m = freqs[j_idx] + freqs[k_idx] - fi

            # Find matching m indices
            # shape: (num_pairs, n)
            matches = target_m[:, None] == freqs[None, :]
            has_match = jnp.any(matches, axis=1)
            m_idx = jnp.argmax(matches, axis=1)

            valid = (
                has_match
                & (j_idx != i)
                & (k_idx != m_idx)
                & (k_idx != i)
                & (j_idx != m_idx)
                & (j_idx <= k_idx)
            )

            idx_i = jnp.stack([
                jnp.full_like(j_idx[valid], i),
                j_idx[valid],
                k_idx[valid],
                m_idx[valid]
            ], axis=-1)

            all_idx.append(idx_i)
            counts.append(idx_i.shape[0])

        M = max(counts) if counts else 0

        idx_pad = []
        valid_pad = []

        for idx_i in all_idx:
            pad_len = M - idx_i.shape[0]
            idx_padded = jnp.pad(idx_i, ((0, pad_len), (0, 0)))
            valid_mask = jnp.concatenate([
                jnp.ones(idx_i.shape[0], dtype=bool),
                jnp.zeros(pad_len, dtype=bool)
            ])
            idx_pad.append(idx_padded)
            valid_pad.append(valid_mask)

        idx_pad = jnp.stack(idx_pad, axis=0)
        valid = jnp.stack(valid_pad, axis=0)

        return idx_pad, valid


    @jax.jit
    def _eta_GN_FWM(Ptot, P, beta2, beta3, beta4, a, a_bar, f, B, Cr, gamma, L, idx_pad, valid):
        def _ch(i):
            idx_ch = idx_pad[i]
            valid_ch = valid[i]
            def _eta_per_ch(Ptot, P, beta2, beta3, beta4, a, a_bar, f, B, Cr, gamma, L, idx_ch, valid_ch):
                T_tilde = -((Ptot * Cr) / (2 * a)) * f
                T = 1 + T_tilde

                def _eta(idx):
                    i, j, k, m = idx

                    def _phi(i, j, k, f, beta2, beta3, beta4):
                        phi_jk  = -4 * pi ** 2 * (f[j] - f[i]) * (f[k] - f[i]) * (beta2 + pi * beta3 * (f[j] + f[k]) + \
                                (2 / 3) * pi ** 2 * beta4 * (f[j] ** 2 + f[j] * f[k] + f[k] ** 2 + 0.5 * (f[j] - f[i]) * (f[k] - f[i])))
                        dphi_f1 = -4 * pi ** 2 * (f[k] - f[i]) * (beta2 + pi * beta3 * (f[j] + f[k] + f[j] - f[i]) + \
                                (2 / 3) * pi ** 2 * beta4 * (f[j] ** 2 + f[j] * f[k] + f[k] ** 2 + 0.5 * (f[j] - f[i]) * (f[k] - f[i]) + (f[j] - f[i]) * (2 * (f[j] - f[i]) + 1.5 * (f[k] - f[i]) + 3 * f[i])))
                        dphi_f2 = -4 * pi ** 2 * (f[j] - f[i]) * (beta2 + pi * beta3 * (f[j] + f[k] + f[k] - f[i]) + \
                                (2 / 3) * pi ** 2 * beta4 * (f[j]**2 + f[j] * f[k] + f[k] ** 2 + 0.5 * (f[j] - f[i]) * (f[k] - f[i]) + (f[k] - f[i]) * (2 * (f[k] - f[i]) + 1.5 * (f[j] - f[i]) + 3 * f[i])))
                        return phi_jk, dphi_f1, dphi_f2

                    phi_jk, dphi_f1, dphi_f2 = _phi(i, j, k, f, beta2, beta3, beta4)

                    def _island(i, j, k, m, a, a_bar, T, T_tilde,
                            B, P, gamma, L, phi_jk, dphi_f1, dphi_f2):
                        Omega = jnp.where(j == k, 1, 2)

                        Bi = B[i]; Bj = B[j]; Bk = B[k]; Bm = B[m]
                        Pi = P[i]; Pj = P[j]; Pk = P[k]; Pm = P[m]

                        gamma_i = gamma[i]

                        f1bound = Bj / 2
                        f2min = -Bk / 2
                        f2max = Bk / 2

                        def _F(a, b, d, f2):
                            term_plus = a + b * f2 + d
                            term_minus = a + b * f2 - d
                            return ((term_plus * arctan(term_plus) - term_minus * arctan(term_minus)) / b
                                - 0.5 / b * (log(1.0 + term_plus**2) - log(1.0 + term_minus**2)))

                        def _island_COI(x):
                            l_j, l_k, l_j_line, l_k_line = x

                            alpha_j = a[j] / 2 + l_j * a_bar[j]
                            alpha_k = a[k] / 2 + l_k * a_bar[k]
                            alpha_all = alpha_j + alpha_k

                            alpha_j_line = a[j] / 2 + l_j_line * a_bar[j]
                            alpha_k_line = a[k] / 2 + l_k_line * a_bar[k]
                            alpha_all_line = alpha_j_line + alpha_k_line

                            T_tilde_j = T[j] * (-T_tilde[j] / T[j]) ** l_j
                            T_tilde_k = T[k] * (-T_tilde[k] / T[k]) ** l_k
                            T_tilde_all = T_tilde_j * T_tilde_k

                            T_tilde_j_line = T[j] * (-T_tilde[j] / T[j]) ** l_j_line
                            T_tilde_k_line = T[k] * (-T_tilde[k] / T[k]) ** l_k_line
                            T_tilde_all_line = T_tilde_j_line * T_tilde_k_line

                            alpha_tilde_all = (alpha_all * (1 - exp(-alpha_all * L))
                                / (1 - exp(-alpha_all * L) - alpha_all * L * exp(-alpha_all * L)))
                            kappa_all = ((1 - exp(-alpha_all * L)) ** 2
                                / (1 - exp(-alpha_all * L) - alpha_all * L * exp(-alpha_all * L)))

                            alpha_tilde_all_line = (alpha_all_line * (1 - exp(-alpha_all_line * L))
                                / (1 - exp(-alpha_all_line * L) - alpha_all_line * L * exp(-alpha_all_line * L)))
                            kappa_all_line = ((1 - exp(-alpha_all_line * L)) ** 2
                                / (1 - exp(-alpha_all_line * L) - alpha_all_line * L * exp(-alpha_all_line * L)))

                            a1 = phi_jk / alpha_tilde_all
                            a2 = phi_jk / alpha_tilde_all_line
                            b1 = dphi_f2 / alpha_tilde_all
                            b2 = dphi_f2 / alpha_tilde_all_line

                            d1 = dphi_f1 * f1bound / alpha_tilde_all
                            d2 = dphi_f1 * f1bound / alpha_tilde_all_line

                            F1 = _F(a1, b1, d1, f2max) - _F(a1, b1, d1, f2min)
                            F2 = _F(a2, b2, d2, f2max) - _F(a2, b2, d2, f2min)

                            term = (T_tilde_all * T_tilde_all_line * kappa_all * kappa_all_line
                                / (alpha_tilde_all + alpha_tilde_all_line) * (F1 + F2) / dphi_f1)
                            return term

                        def _island_FWM(x):
                            l_j, l_k, l_m, l_j_line, l_k_line, l_m_line = x

                            alpha_j = a[j] / 2 + l_j * a_bar[j]
                            alpha_k = a[k] / 2 + l_k * a_bar[k]
                            alpha_m = a[m] / 2 + l_m * a_bar[m]
                            alpha_i = a[i] / 2
                            alpha_all = alpha_j + alpha_k + alpha_m - alpha_i

                            alpha_j_line = a[j] / 2 + l_j_line * a_bar[j]
                            alpha_k_line = a[k] / 2 + l_k_line * a_bar[k]
                            alpha_m_line = a[m] / 2 + l_m_line * a_bar[m]
                            alpha_i_line = a[i] / 2
                            alpha_all_line = alpha_j_line + alpha_k_line + alpha_m_line - alpha_i_line

                            T_tilde_j = T[j] * (-T_tilde[j] / T[j]) ** l_j
                            T_tilde_k = T[k] * (-T_tilde[k] / T[k]) ** l_k
                            T_tilde_m = T[m] * (-T_tilde[m] / T[m]) ** l_m
                            T_tilde_all = T_tilde_j * T_tilde_k * T_tilde_m

                            T_tilde_j_line = T[j] * (-T_tilde[j] / T[j]) ** l_j_line
                            T_tilde_k_line = T[k] * (-T_tilde[k] / T[k]) ** l_k_line
                            T_tilde_m_line = T[m] * (-T_tilde[m] / T[m]) ** l_m_line
                            T_tilde_all_line = T_tilde_j_line * T_tilde_k_line * T_tilde_m_line

                            alpha_tilde_all = (alpha_all * (1 - exp(-alpha_all * L))
                                / (1 - exp(-alpha_all * L) - alpha_all * L * exp(-alpha_all * L)))
                            kappa_all = ((1 - exp(-alpha_all * L)) ** 2
                                / (1 - exp(-alpha_all * L) - alpha_all * L * exp(-alpha_all * L)))

                            alpha_tilde_all_line = (alpha_all_line * (1 - exp(-alpha_all_line * L))
                                / (1 - exp(-alpha_all_line * L) - alpha_all_line * L * exp(-alpha_all_line * L)))
                            kappa_all_line = ((1 - exp(-alpha_all_line * L)) ** 2
                                / (1 - exp(-alpha_all_line * L) - alpha_all_line * L * exp(-alpha_all_line * L)))

                            a1 = phi_jk / alpha_tilde_all
                            a2 = phi_jk / alpha_tilde_all_line
                            b1 = dphi_f2 / alpha_tilde_all
                            b2 = dphi_f2 / alpha_tilde_all_line

                            d1 = dphi_f1 * f1bound / alpha_tilde_all
                            d2 = dphi_f1 * f1bound / alpha_tilde_all_line

                            F1 = _F(a1, b1, d1, f2max) - _F(a1, b1, d1, f2min)
                            F2 = _F(a2, b2, d2, f2max) - _F(a2, b2, d2, f2min)

                            term = (T_tilde_all * T_tilde_all_line * kappa_all * kappa_all_line
                                / (alpha_tilde_all + alpha_tilde_all_line) * (F1 + F2) / dphi_f1)
                            return term

                        sum = jax.lax.cond(
                            m == i,
                            lambda _: jax.vmap(_island_COI)(_INDICES_COI).sum(axis=0),
                            lambda _: jax.vmap(_island_FWM)(_INDICES_FWM).sum(axis=0),
                            operand=None
                        )

                        eta_jkm = nan_to_num(
                            Omega * (16 / 27) * gamma_i ** 2 * (Bi / Pi ** 3)
                            * (Pj * Pk * Pm / (Bj * Bk * Bm)) * sum,
                            nan=0,
                            posinf=0,
                            neginf=0
                            )

                        return eta_jkm

                    return _island(i, j, k, m, a, a_bar, T, T_tilde, B, P, gamma, L, phi_jk, dphi_f1, dphi_f2)

                return jnp.where(valid_ch, jax.vmap(_eta)(idx_ch).squeeze(), 0).sum(axis=0)
            
            return _eta_per_ch(Ptot, P, beta2, beta3, beta4, a, a_bar, f, B, Cr, gamma, L, idx_ch, valid_ch)
        
        return jax.lax.map(_ch, jnp.arange(f.size))

    _eta_SPM = _eta_GN_SPM(phi_i, B_i, a_i, a_bar_i, gamma_ij, Tf_i, T_i, setup.length_j[j])
    _eta_XPM = _eta_GN_XPM(P_i, P_k, phi_ik, B_i, B_k, a_k, a_bar_k, gamma_ij, Tf_k, T_k, setup.length_j[j])

    if ch_idx_oband is not None:
        idx_pad, valid = _FWM_idx(f_i[ch_idx_oband])
        _eta_FWM = jnp.zeros_like(_eta_XPM)
        _eta_FWM = _eta_FWM.at[ch_idx_oband].set(_eta_GN_FWM(Ptot, P_i[ch_idx_oband], beta2_j[j], beta3_j[j], beta4_j[j], a_i[ch_idx_oband],
                                                            a_bar_i[ch_idx_oband], f_i[ch_idx_oband], B_i[ch_idx_oband], Cr_i[ch_idx_oband],
                                                            gamma_ij[ch_idx_oband, j], setup.length_j[j], idx_pad, valid))
    else:
        idx_pad, valid = _FWM_idx(f_i)
        _eta_FWM = _eta_GN_FWM(Ptot, P_i, beta2_j[j], beta3_j[j], beta4_j[j], a_i, a_bar_i, f_i, B_i, Cr_i, gamma_ij[:, j], setup.length_j[j], idx_pad, valid)

    # Linear on/off gain for egn.calc_Pase (see ong/examples/egn_example.py); not dB.
    _tiny = jnp.array(1e-30, dtype=power_evo.dtype)
    ratio_in_out = power_evo[0, :] / jnp.maximum(power_evo[-1, :], _tiny)
    gain_lin = jnp.maximum(ratio_in_out, jnp.array(1.0 + 1e-9, dtype=power_evo.dtype))
    p_ASE = egn.calc_Pase(
                gain_lin,
                setup.NF_i[chs],
                ref_lambda=setup.ref_lambda,
                ch_centre_ij=setup.ch_centre_ij[chs, :],
                ch_bandwidth_ij=setup.ch_bandwidth_ij[chs, :]
            ) * Nspans

    return Nspans * P_i ** 3 * (_eta_SPM[:, None] + _eta_XPM[:, None] + _eta_FWM[:, None]) / P_i + p_ASE[:, None] / P_i + 1 / idB(setup.snr_trx[chs, None])

In [10]:
import json
import numpy as np
import jax.numpy as jnp

def path_to_edges(path):
    return [(path[i], path[i + 1]) for i in range(len(path) - 1)]


def rwa_throughput(topology_json, rwa_json, setup, calc_NSR_link, ch_idx_oband=None, sort_edges=True):
    """
    Compute total RWA throughput from topology + RWA + link NSR model.

    Parameters
    ----------
    topology_json : str or dict
    rwa_json : str or dict
    setup : object
        Passed into calc_NSR_link(setup, Nspans, mask)
    calc_NSR_link : callable
        Function with signature:
            calc_NSR_link(setup, Nspans, mask) -> per-channel NSR
        Expected output shape: (num_wavelengths,) or (num_wavelengths, 1)
    ch_bandwidth : float, optional
        Channel bandwidth in Hz.
        If None, uses setup.ch_bandwidth_ij[0,0] if available.
    sort_edges : bool

    Returns
    -------
    total_throughput : float
        Sum of all lightpath Shannon capacities [bit/s]

    throughput_per_lightpath : list[dict]
        Per-lightpath details:
        {
            "wavelength": int,
            "node_path": [...],
            "edge_path": [...],
            "nsr": float,
            "snr": float,
            "rate_bps": float
        }

    nsr_link_channel : jnp.ndarray
        Shape (num_links, num_wavelengths)

    occupancy_matrix : np.ndarray
        Shape (num_links, num_wavelengths)
    """

    if isinstance(topology_json, str):
        with open(topology_json, "r") as f:
            topology = json.load(f)
    else:
        topology = topology_json

    if isinstance(rwa_json, str):
        with open(rwa_json, "r") as f:
            rwa = json.load(f)
    else:
        rwa = rwa_json

    if not topology or "links" not in topology:
        raise ValueError("Invalid topology_json: missing 'links' field.")
    if not rwa:
        raise ValueError("Invalid rwa_json: RWA data is empty.")

    def canon_edge(u, v):
        return (min(u, v), max(u, v))

    # topology -> edge/span map
    edge_span_map = {}
    for link in topology["links"]:
        u = link["u"]
        v = link["v"]
        spans = link.get("spans", 1)
        edge_span_map[canon_edge(u, v)] = spans

    edges = list(edge_span_map.keys())
    if sort_edges:
        edges = sorted(edges)

    edge_to_row = {edge: i for i, edge in enumerate(edges)}
    span_count_per_edge = np.array([edge_span_map[e] for e in edges], dtype=int)

    wavelengths = sorted(int(w) for w in rwa.keys())
    wavelength_to_col = {w: i for i, w in enumerate(wavelengths)}

    # build occupancy matrix + edge-path rwa
    occupancy_matrix = np.zeros((len(edges), len(wavelengths)), dtype=int)
    rwa_edge_paths = {}

    for w_str, paths in rwa.items():
        w = int(w_str)
        col = wavelength_to_col[w]
        rwa_edge_paths[w_str] = []

        for node_path in paths:
            if not node_path or len(node_path) < 2:
                continue

            edge_path = [canon_edge(u, v) for u, v in path_to_edges(node_path)]
            rwa_edge_paths[w_str].append({
                "node_path": node_path,
                "edge_path": edge_path,
            })

            for edge in edge_path:
                if edge in edge_to_row:
                    occupancy_matrix[edge_to_row[edge], col] = 1

    # compute per-link per-channel NSR
    nsr_link_channel = jnp.zeros_like(jnp.asarray(occupancy_matrix), dtype=jnp.float64)

    for i in range(span_count_per_edge.shape[0]):
        link_nsr = calc_NSR_link(
            setup,
            span_count_per_edge[i],
            occupancy_matrix[i, :, None],
            ch_idx_oband,
        ).squeeze(-1)
        nsr_link_channel = nsr_link_channel.at[i].set(link_nsr)

    # compute throughput by summing over all lightpaths
    throughput_per_lightpath = []
    total_throughput = 0.0

    for w_str, path_infos in rwa_edge_paths.items():
        w = int(w_str)
        col = wavelength_to_col[w]

        for info in path_infos:
            edge_path = info["edge_path"]

            path_nsr = 0.0
            for edge in edge_path:
                row = edge_to_row[edge]
                path_nsr += float(nsr_link_channel[row, col])

            if path_nsr <= 0:
                snr = np.inf
                rate_bps = np.inf
            else:
                snr = 1.0 / path_nsr
                rate_bps = 2 * setup.ch_bandwidth_ij[col, 0] * np.log2(1.0 + snr)

            throughput_per_lightpath.append({
                "wavelength": w,
                "node_path": info["node_path"],
                "edge_path": edge_path,
                "nsr": path_nsr,
                "snr": snr,
                "rate_bps": rate_bps,
            })

            total_throughput += rate_bps

    return total_throughput, throughput_per_lightpath, nsr_link_channel, occupancy_matrix

In [12]:
topology_fname = 'rwa_878.json'
# rwa_fname = 'rwa_simulation_results.json'  # 236
# rwa_fname = 'SCLO_simulation_results.json'  # 920
rwa_fname = 'rwa_878n.json'  # 878

total_throughput, lp_info, nsr_link_channel, occupancy_matrix = rwa_throughput(
    topology_json=topology_fname,
    rwa_json=rwa_fname,
    setup=setup,
    calc_NSR_link=calc_NSR_link,
    ch_idx_oband=ch_idx_oband[channel_idx]
)

/tmp/ipykernel_3413864/1511378061.py:113: UserWarning: Explicitly requested dtype float64 requested in zeros_like is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  nsr_link_channel = jnp.zeros_like(jnp.asarray(occupancy_matrix), dtype=jnp.float64)


In [ ]:
total_throughput / 1e12

In [ ]:
plt.plot(setup.ch_lambda_ij[channel_idx] * 1e9, dB(1 / nsr_link_channel[5, :]), '.')
plt.grid()
plt.xlabel("Wavelength [nm]")
plt.ylabel("SNR [dB]")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.imshow(occupancy_matrix, cmap="gray_r", aspect="auto", interpolation="none")

# grid on every cell
ax = plt.gca()
ax.set_xticks(np.arange(-0.5, occupancy_matrix.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-0.5, occupancy_matrix.shape[0], 1), minor=True)
# ax.grid(which="minor", color="black", linestyle="-", linewidth=0.5)

ax.tick_params(which="minor", bottom=False, left=False)

plt.xlabel("Channel")
plt.ylabel("Link")
plt.show()